# Avance 4: modelos alternativos para estimar la emisión de NOₓ

## 1. Objetivo

Comparar modelos con **algoritmos distintos** que estiman la misma cantidad, la emisión de NOₓ de la Ciudad de México por zona y tipo de día, con los mismos datos (noviembre–diciembre de 2023, tardes), el mismo esquema de validación (una estación oculta a la vez) y las mismas pruebas de cordura (placebo del domingo). Con la tabla se eligen los dos mejores, se ajustan con una búsqueda de hiperparámetros y se argumenta cuál es el modelo final.

Reglas del ejercicio:

- Todos los modelos **estiman emisión**; IDW y kriging aparecen solo como piso de concentración.
- Ningún modelo ve febrero–mayo de 2024: es el periodo de prueba del avance 5. `nox-ml tune` rechaza configuraciones que lo incluyan.
- Todo el código de este cuaderno llama a `nox_ml`; los números salen de las carpetas `runs/` (`nox-ml run`, `nox-ml compare`, `nox-ml tune`).

## 2. Los modelos y las referencias

| # | Modelo (`nox_ml`) | Algoritmo | Por qué está en la comparación | Emisión que estima |
|---|---|---|---|---|
| E1 | `inversion_plume` | Inversión bayesiana lineal con pluma gaussiana estacionaria (**baseline**) | Punto de partida: física simple e interpretable, con incertidumbre. Fija lo que los demás deben mejorar | Absoluta, por zona y tipo de día |
| E2 | `inversion_puff` | Inversión bayesiana lineal con puffs y vida química por OH | Quita la hipótesis de estado estacionario: el aire conserva emisiones de horas anteriores | Absoluta |
| E3 | `regularized_ls` | Mínimos cuadrados regularizados y no negativos (ridge/NNLS) sobre la tabla física | Misma física, sin supuesto bayesiano: aísla el efecto del prior | Absoluta |
| E4 | `lognormal_map` | Inversión no lineal con emisiones log-normales (siempre positivas): máximo a posteriori con L-BFGS-B e incertidumbre por aproximación de Laplace | Garantiza emisiones positivas sin truncar y prueba un prior que no es gaussiano | Absoluta |
| E5a | `pinn_plume` | Red neuronal informada por la física (PINN) que aprende el núcleo de advección–difusión + inversión bayesiana | Cambia el transporte analítico por uno aprendido, sin cambiar la inversión | Absoluta |
| E5b | `pinn_puff` | PINN aplicada a cada tramo de puff + inversión bayesiana | Combina el núcleo aprendido con la memoria de los puffs | Absoluta |
| E5c | `pinn_joint` | PINN entrenada con la física y las estaciones a la vez | Deja que los datos corrijan el transporte (se valida con 4 grupos de estaciones por costo) | Absoluta |
| E6 | `rf_deweather` | Normalización meteorológica con bosque aleatorio (concentración ~ meteorología + calendario; la emisión es lo que queda al fijar la meteorología) | Algoritmo sin física, de otra familia: no depende del modelo de transporte | **Relativa** (índice por tipo de día) |
| E7 | `normalized_ratio` | Estimador normalizado: observado ÷ esperado con el modelo de transporte y emisión fija | Es el estimador más simple del cambio de emisión | **Relativa** |
| R1, R2 | `idw`, `kriging` | Interpolación espacial (referencias) | Piso de concentración: un modelo físico que no los iguale no aporta | — (no estiman emisión) |

> **[POR ESCRIBIR — equipo]** Una o dos frases por modelo con la intuición de sus hiperparámetros principales y las decisiones de implementación que no son obvias (ver ADR 0004, 0005 y 0006 para las PINN y el bosque aleatorio).

## 3. Tabla comparativa

La tabla sale solo de `nox_ml.validation.compare` aplicado a las corridas de `runs/` (sin las corridas de ajuste fino, que se ven en la sección 5). Está ordenada por el error con estación oculta (`loo_rmse_ppb`, ppb; menor es mejor) e incluye el sesgo, el error horario, la correlación con TEMPO, el cociente domingo/hábil con su intervalo, el placebo, la elegibilidad (el centro del placebo debe estar en 1 ± 0.15) y los tiempos.

In [ ]:
from nox_ml.validation import compare, report

RUNS = report.find_runs()
table, skipped = compare.collect(report.run_dirs(RUNS))
print(f"{len(table)} corridas en {RUNS}")
for path in skipped:
    print(f"  omitida {path.name}: sin resumen en metrics.json")

Modelos previstos que **todavía no tienen corrida** (con el comando que la produce):

In [ ]:
pending = report.missing(table)
print("Todos los modelos previstos tienen corrida." if pending.empty else f"Faltan {len(pending)} modelos:")
pending

In [ ]:
table.round(2)

> **[POR ESCRIBIR — equipo, con los resultados finales]** Leer la tabla: qué modelos quedan por debajo de IDW y kriging en concentración; qué modelos no pasan el placebo (columna `eligible`) y por qué no compiten; qué tan lejos queda el domingo/hábil de 0.65–0.73 (columna `sunday_ratio_gap`); relación entre costo (`fit_seconds`) y error.

## 4. Los dos mejores

Compiten los modelos (no las referencias) que pasan el placebo; se eligen los dos de menor `loo_rmse_ppb`.

In [ ]:
best = report.selection(table, n=2)
print(f"Elegibles seleccionados: {len(best)} de 2")
best[
    ["run", "model", "loo_rmse_ppb", "sunday_ratio", "placebo_center", "placebo_halfwidth", "fit_seconds"]
].round(2)

> **[POR ESCRIBIR — equipo]** Cuáles son los dos y por qué (la tabla decide; se esperaba E2 y uno entre E4, E5 y E6). Si hay modelos con error similar, decir por qué se prefiere uno.

## 5. Ajuste fino

Cada uno de los dos se ajusta con `nox-ml tune` sobre noviembre–diciembre de 2023: búsqueda aleatoria (o en grilla) sobre sus hiperparámetros, con el error con estación oculta como métrica de cada prueba, y reajuste de la mejor. La búsqueda incluye las mejoras de la investigación: incertidumbre del viento, peso del satélite, restricción de tráfico, viento mínimo, zonas administrativas, supuestos de centrales y horas de las 17:00.

**Advertencia:** `nox-ml tune` usa validación simple (no anidada), así que el error ajustado es algo optimista.

In [ ]:
trials = report.read_trials(RUNS)
print("Búsquedas encontradas:", ", ".join(trials) if trials else "ninguna (todavía no se corrió nox-ml tune)")
report.tuning_summary(RUNS, table).round(2)

**Espacio de búsqueda y mejor configuración** de cada búsqueda:

In [ ]:
for run, frame in trials.items():
    print(f"== {run}: {len(frame)} pruebas ==")
    print(report.search_space(frame).to_string(index=False))
    print("Mejor prueba:")
    print(report.best_trial(frame).to_string())
if not trials:
    print("Sin pruebas que mostrar.")

**Curva de pruebas:** el error de cada prueba, el mejor hasta ese momento y el modelo sin ajustar.

In [ ]:
report.plot_trials(RUNS, table)

> **[POR ESCRIBIR — equipo]** Mejora obtenida frente al modelo sin ajustar (ppb y %), si la curva se aplana (la búsqueda fue suficiente) o sigue bajando, qué hiperparámetros movieron más el error, y si el reajuste de la mejor configuración sigue pasando el placebo.

## 6. Modelo final

La métrica principal no basta para elegir. Los argumentos que cuentan:

- error con estación oculta (`loo_rmse_ppb`);
- exactitud del cambio de emisión: cociente domingo/hábil frente al rango del tráfico contado, con su intervalo;
- placebos: centro cerca de 1 y semiancho (el cambio mínimo que el modelo podría detectar);
- interpretabilidad (factores por zona);
- incertidumbre de la emisión estimada;
- emisión absoluta o solo relativa;
- costo de ajuste y de predicción.

In [ ]:
final = report.selection(table, n=2)
final[
    [
        "run",
        "model",
        "loo_rmse_ppb",
        "sunday_ratio",
        "sunday_ratio_lo",
        "sunday_ratio_hi",
        "sunday_ratio_gap",
        "placebo_center",
        "placebo_halfwidth",
        "fit_seconds",
        "predict_seconds",
        "n_params",
    ]
].round(2)

> **[POR ESCRIBIR — equipo, con los resultados finales]** Modelo final elegido y los argumentos, uno por viñeta de la lista anterior: qué modelo gana en cada criterio, qué se sacrifica y por qué se acepta. Incluir las limitaciones (validación simple en el ajuste, ausencia de periodo de prueba hasta el avance 5, E5c validada con 4 grupos de estaciones si se corrió).

## 7. Declaración del uso de herramientas de IA

> **[POR ESCRIBIR — equipo]** Declaración obligatoria. Indicar qué herramientas de IA se usaron, para qué partes del trabajo (código, pruebas, redacción, revisión), cómo se verificó lo que produjeron y qué decisiones de método tomó el equipo por su cuenta.